In [1]:
from openai import OpenAI

client = OpenAI()

In [7]:
import sys
!uv pip install --python "{sys.executable}" pandas

Using Python 3.14.2 environment at: /workspaces/AI--Engineering-course-codespace/.venv
Resolved 4 packages in 635ms                                         
Prepared 1 package in 258ms                                                  pandas               ------------------------------ 9.27 MiB/10.28 MiB          
░░░░░░░░░░░░░░░░░░░░ [0/1] Installing wheels...                                 warning: Failed to hardlink files; falling back to full copy. This may lead to degraded performance.
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 1 package in 343ms                                
 + pandas==3.0.6


In [9]:
import pandas as pd

books = pd.read_csv("books.csv")
books.head()

FileNotFoundError: [Errno 2] No such file or directory: 'books.csv'

In [10]:
import pandas as pd

books = pd.read_csv(
    "/workspaces/AI--Engineering-course-codespace/01 -foundation /books.csv"
)
books.head()

,title,book_url,pdf_url
0,Think Python 2e,https://greenteapress.com/wp/think-python-2e/,http://greenteapress.com/thinkpython2/thinkpyt...
1,Think DSP,https://greenteapress.com/wp/think-dsp/,http://greenteapress.com/thinkdsp/thinkdsp.pdf
2,Think Complexity 2e,https://greenteapress.com/wp/think-complexity/,http://greenteapress.com/complexity2/thinkcomp...
3,Think Java 2e,https://greenteapress.com/wp/think-java-2e/,http://greenteapress.com/thinkjava7/thinkjava2...
4,Physical Modeling in MATLAB,https://greenteapress.com/wp/physical-modeling...,https://github.com/AllenDowney/PhysicalModelin...


In [11]:
import pandas as pd
import requests
from pathlib import Path

Path("books").mkdir(exist_ok=True)

df = pd.read_csv(
    "/workspaces/AI--Engineering-course-codespace/01 -foundation /books.csv"
)

for _, row in df.iterrows():
    url = row["pdf_url"]
    filename = url.split("/")[-1]
    dest = Path("books") / filename

    if dest.exists():
        print(f"Skipping {filename} (already exists)")
        continue

    print(f"Downloading {filename}...")
    response = requests.get(url)
    response.raise_for_status()
    dest.write_bytes(response.content)
    print(f"Saved {filename}")

Saved thinkpython2.pdf
Saved thinkdsp.pdf
Saved thinkcomplexity2.pdf
Saved thinkjava2.pdf
Saved PhysicalModelingInMatlab4.pdf
Saved thinkos.pdf
Saved Think-C.pdf


In [12]:
from pathlib import Path
from markitdown import MarkItDown

books_folder = Path("books")
converter = MarkItDown()

for pdf in books_folder.glob("*.pdf"):
    destination = pdf.with_suffix(".md")

    if destination.exists():
        print(f"Skipping {destination.name} (already exists)")
        continue

    try:
        print(f"Converting {pdf.name}...")

        result = converter.convert_local(str(pdf))
        destination.write_text(result.text_content, encoding="utf-8")

        print(f"Saved {destination.name}")

    except Exception as error:
        print(f"Failed: {pdf.name} — {error}")

print(f"Files saved in: {books_folder.resolve()}")

2026-10-08 14:18:19.130259926 [W:onnxruntime:Default, device_discovery.cc:146 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


Converting thinkcomplexity2.pdf...
Saved thinkcomplexity2.md
Converting thinkos.pdf...
Saved thinkos.md
Converting thinkjava2.pdf...
Saved thinkjava2.md
Converting thinkdsp.pdf...
Saved thinkdsp.md
Converting PhysicalModelingInMatlab4.pdf...
Saved PhysicalModelingInMatlab4.md
Converting Think-C.pdf...
Saved Think-C.md
Converting thinkpython2.pdf...
Saved thinkpython2.md
Files saved in: /workspaces/AI--Engineering-course-codespace/01 -foundation /02-rag/books


In [16]:
!wc -l books/thinkpython2.md

9702 books/thinkpython2.md


In [17]:
from pathlib import Path

documents = []

for file in sorted(Path("books").glob("*.md")):
    text = file.read_text(encoding="utf-8")
    lines = [line for line in text.splitlines() if line.strip()]

    documents.append({
        "source": file.name,
        "content": lines,
    })

print(f"Prepared {len(documents)} books")

Prepared 7 books


In [18]:
from pathlib import Path
from gitsource import chunk_documents

file = Path("books/thinkpython2.md")
text = file.read_text(encoding="utf-8")

documents = [{
    "source": file.name,
    "content": [line for line in text.splitlines() if line.strip()],
}]

chunks = chunk_documents(documents, size=100, step=50)

print(len(chunks))  # 189

189


In [22]:
from pathlib import Path
from gitsource import chunk_documents
from minsearch import Index

documents = []

for file in sorted(Path("books").glob("*.md")):
    text = file.read_text(encoding="utf-8")

    documents.append({
        "source": file.name,
        "content": [
            line for line in text.splitlines() if line.strip()
        ],
    })

chunks = chunk_documents(documents, size=100, step=50)

search_documents = [
    {**chunk, "content": "\n".join(chunk["content"])}
    for chunk in chunks
]

index = Index(
    text_fields=["content"],
    keyword_fields=["source"],
)
index.fit(search_documents)

print(f"Indexed {len(search_documents)} chunks")  # 988

Indexed 988 chunks


In [25]:
results = index.search("python function definition", num_results=5)

In [26]:
for rank, result in enumerate(results, start=1):
    print(f"{rank}. {result['source']}")

1. PhysicalModelingInMatlab4.md
2. thinkdsp.md
3. thinkpython2.md
4. PhysicalModelingInMatlab4.md
5. PhysicalModelingInMatlab4.md


In [27]:
import json

instructions = """
You're a course assistant, your task is to answer the QUESTION from the
course students using the provided CONTEXT
"""

prompt_template = """
<QUESTION>
{question}
</QUESTION>

<CONTEXT>
{context}
</CONTEXT>
""".strip()


def build_prompt(question, search_results):
    context = json.dumps(search_results, indent=2)
    prompt = prompt_template.format(
        question=question,
        context=context,
    ).strip()
    return prompt


def search(question):
    return index.search(question, num_results=5)


def llm(user_prompt, instructions, model="gpt-4o-mini"):
    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": user_prompt},
    ]

    response = openai_client.responses.create(
        model=model,
        input=messages,
    )

    return {
        "answer": response.output_text,
        "input_tokens": response.usage.input_tokens,
        "output_tokens": response.usage.output_tokens,
    }


def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    return llm(prompt, instructions)


result = rag("python function definition")

print(result["answer"])
print("\nInput tokens:", result["input_tokens"])
print("Output tokens:", result["output_tokens"])

NameError: name 'openai_client' is not defined

In [28]:
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
openai_client = OpenAI()

In [29]:
import json

instructions = """
You're a course assistant, your task is to answer the QUESTION from the
course students using the provided CONTEXT
"""

prompt_template = """
<QUESTION>
{question}
</QUESTION>

<CONTEXT>
{context}
</CONTEXT>
""".strip()


def build_prompt(question, search_results):
    context = json.dumps(search_results, indent=2)
    prompt = prompt_template.format(
        question=question,
        context=context,
    ).strip()
    return prompt


def search(question):
    return index.search(question, num_results=5)


def llm(user_prompt, instructions, model="gpt-4o-mini"):
    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": user_prompt},
    ]

    response = openai_client.responses.create(
        model=model,
        input=messages,
    )

    return {
        "answer": response.output_text,
        "input_tokens": response.usage.input_tokens,
        "output_tokens": response.usage.output_tokens,
    }


def rag(query):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    return llm(prompt, instructions)


result = rag("python function definition")

print(result["answer"])
print("\nInput tokens:", result["input_tokens"])
print("Output tokens:", result["output_tokens"])

To define a function in Python, you start with the `def` keyword followed by the function name and parentheses that contain any parameters. The function body is indented under the definition. Here's a simple example:

```python
def my_function(x):
    result = x * 2
    return result
```

In this example:
- `my_function` is the name of the function.
- `x` is the parameter that the function takes as input.
- The function calculates `result` as `x * 2` and returns it.

To call the function, you would use its name followed by parentheses containing any arguments:

```python
output = my_function(5)
print(output)  # This will output 10
```

This method of defining functions helps to prevent variable name collisions and creates a separate workspace for the function variables, which are not accessible outside the function. Thus, the performance and management of code become more organized.

Input tokens: 7926
Output tokens: 194


In [30]:
query = "python function definition"
prompt = build_prompt(query, search(query))

count = openai_client.responses.input_tokens.count(
    model="gpt-4o-mini",
    input=[
        {"role": "system", "content": instructions},
        {"role": "user", "content": prompt},
    ],
)

print("Input tokens:", count.input_tokens)

Input tokens: 7926


In [31]:
from pydantic import BaseModel, Field
from typing import Literal


class RAGResponse(BaseModel):
    answer: str = Field(
        description="The main answer to the user's question in markdown"
    )
    found_answer: bool = Field(
        description="True if relevant information was found in the documentation"
    )
    confidence: float = Field(
        description="Confidence score from 0.0 to 1.0"
    )
    confidence_explanation: str = Field(
        description="Explanation about the confidence level"
    )
    answer_type: Literal[
        "how-to", "explanation", "troubleshooting", "comparison", "reference"
    ] = Field(description="The category of the answer")
    followup_questions: list[str] = Field(
        description="Suggested follow-up questions"
    )


def llm(user_prompt, instructions, model="gpt-4o-mini", structured=False):
    messages = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": user_prompt},
    ]

    if structured:
        response = openai_client.responses.parse(
            model=model,
            input=messages,
            text_format=RAGResponse,
        )
        answer = response.output_parsed
    else:
        response = openai_client.responses.create(
            model=model,
            input=messages,
        )
        answer = response.output_text

    return {
        "answer": answer,
        "input_tokens": response.usage.input_tokens,
        "output_tokens": response.usage.output_tokens,
    }


def rag(query, structured=False):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    return llm(prompt, instructions, structured=structured)

In [32]:
query = "python function definition"

unstructured = rag(query)
structured = rag(query, structured=True)

print(structured["answer"].model_dump_json(indent=2))

extra_tokens = (
    structured["input_tokens"] - unstructured["input_tokens"]
)

print("\nUnstructured input tokens:", unstructured["input_tokens"])
print("Structured input tokens:", structured["input_tokens"])
print("MORE input tokens:", extra_tokens)

{
  "answer": "### Python Function Definition\n\nIn Python, a function is defined using the `def` keyword followed by the function name and parentheses that may include parameters. The function body is indented and can include any block of code. Here’s a basic example of a function definition:\n\n```python\ndef my_function(param1, param2):\n    \"\"\"This is a docstring that describes the function.\"\"\"\n    result = param1 + param2\n    return result\n```\n\n### Explanation:\n1. **Defining a function**: Start with `def`, followed by the function name.\n2. **Parameters**: Enclosed in parentheses, you can pass values into the function.\n3. **Docstring**: A descriptive comment about what the function does (optional but recommended).\n4. **Function Body**: The code block that performs the task is indented.\n5. **Return Statement**: `return` sends a value back to where the function was called.\n\n### Example of Usage:\n```python\nresult = my_function(10, 5)\nprint(result)  # This will out